In [ ]:
import random
import time
from collections import OrderedDict
from copy import deepcopy
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import display
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
try:
    torch.set_float32_matmul_precision('high')
except AttributeError:
    pass


@dataclass(frozen=True)
class Config:
    # ==============================
    # Required hyperparameters
    # ==============================
    NUM_CLIENTS: int = 20
    BUDGET: float = 100.0
    GLOBAL_ROUNDS: int = 100
    LOCAL_EPOCHS: int = 2
    BATCH_SIZE: int = 32
    LR: float = 0.01
    MC_ITERATIONS: int = 10

    # ==============================
    # Benchmark and runtime settings
    # ==============================
    DIRICHLET_ALPHA: float = 0.5
    VAL_RATIO: float = 0.10
    RANDOM_SEED: int = 42
    DATA_DIR: str = './data'
    DIGITS_FILE: str = 'emnist-digits.mat'
    LETTERS_FILE: str = 'emnist-letters.mat'
    INITIAL_REPUTATION: float = 5.0
    REPUTATION_REWARD: float = 0.5
    REPUTATION_PENALTY: float = 2.0
    HACKER_RATIO: float = 0.20
    BID_ATTACK_RATIO: float = 0.30
    BID_ATTACK_MULTIPLIER: float = 2.40
    NUM_CLASSES_DIGITS: int = 10
    NUM_CLASSES_LETTERS: int = 26
    NUM_WORKERS: int = 0
    MC_VAL_SUBSET_SIZE: Optional[int] = 4000
    MAX_TRAIN_SAMPLES: Optional[int] = None
    MAX_VAL_SAMPLES: Optional[int] = None
    LOG_EVERY: int = 1
    CONVERGENCE_RATIO: float = 0.95

    @property
    def NUM_CLASSES(self) -> int:
        return self.NUM_CLASSES_DIGITS + self.NUM_CLASSES_LETTERS


cfg = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_data_files(config: Config) -> Tuple[Path, Path]:
    # This function supports both local execution and Google Colab.
    data_dir = Path(config.DATA_DIR)
    data_dir.mkdir(parents=True, exist_ok=True)

    digits_path = data_dir / config.DIGITS_FILE
    letters_path = data_dir / config.LETTERS_FILE

    if digits_path.exists() and letters_path.exists():
        print(f'Found dataset files at: {data_dir.resolve()}')
        return digits_path, letters_path

    print('Dataset files were not found. If you are using Colab, upload both .mat files now.')
    try:
        from google.colab import files

        uploaded = files.upload()
        for filename, content in uploaded.items():
            target_path = data_dir / Path(filename).name
            with open(target_path, 'wb') as file_obj:
                file_obj.write(content)
    except ImportError:
        print('This environment is not Google Colab. Please place the files under ./data before running.')

    missing_files = [
        path.name
        for path in (digits_path, letters_path)
        if not path.exists()
    ]
    if missing_files:
        raise FileNotFoundError(
            'Missing dataset files: '
            + ', '.join(missing_files)
            + '. Please upload them to the data/ directory.'
        )

    return digits_path, letters_path


set_seed(cfg.RANDOM_SEED)
digits_path, letters_path = ensure_data_files(cfg)
print(f'Running on device: {device}')
print(f'Total number of classes after merging digits + letters: {cfg.NUM_CLASSES}')

Dataset files were not found. If you are using Colab, upload both .mat files now.


Saving emnist-digits.mat to emnist-digits.mat
Saving emnist-letters.mat to emnist-letters.mat
Running on device: cuda
Total number of classes after merging digits + letters: 36


In [ ]:
IMAGE_KEYS = {'images', 'image', 'x', 'data'}
LABEL_KEYS = {'labels', 'label', 'y', 'target', 'targets'}
CLASS_NAMES = [str(index) for index in range(cfg.NUM_CLASSES_DIGITS)] + [chr(ord('A') + index) for index in range(cfg.NUM_CLASSES_LETTERS)]


def matlab_to_python(obj: Any) -> Any:
    # Recursively convert MATLAB structs into Python-native objects.
    if hasattr(obj, '_fieldnames'):
        return {
            field: matlab_to_python(getattr(obj, field))
            for field in obj._fieldnames
        }
    if isinstance(obj, np.ndarray):
        if obj.dtype == object:
            return [matlab_to_python(item) for item in obj.flat]
        return obj
    if isinstance(obj, list):
        return [matlab_to_python(item) for item in obj]
    if isinstance(obj, tuple):
        return tuple(matlab_to_python(item) for item in obj)
    return obj


def h5_to_python(node: Any, file_obj: h5py.File) -> Any:
    # Fallback loader for MAT files that behave like HDF5 containers.
    if isinstance(node, h5py.Group):
        return {
            key: h5_to_python(node[key], file_obj)
            for key in node.keys()
        }
    if isinstance(node, h5py.Dataset):
        data = node[()]
        ref_dtype = h5py.check_dtype(ref=node.dtype)
        if ref_dtype is not None:
            refs = np.array(data).flatten().tolist()
            return [h5_to_python(file_obj[ref], file_obj) for ref in refs]
        return np.array(data)
    return node


def load_mat_file(mat_path: Path) -> Dict[str, Any]:
    try:
        raw_data = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
        return {
            key: matlab_to_python(value)
            for key, value in raw_data.items()
            if not key.startswith('__')
        }
    except NotImplementedError:
        with h5py.File(mat_path, 'r') as file_obj:
            return {
                key: h5_to_python(file_obj[key], file_obj)
                for key in file_obj.keys()
            }


def standardize_images(images: Any) -> np.ndarray:
    # Convert images into [N, 1, 28, 28] with uint8 dtype to save memory.
    image_array = np.asarray(images)
    image_array = np.squeeze(image_array)

    if image_array.ndim == 2:
        if image_array.shape[1] == 28 * 28:
            image_array = image_array.reshape(-1, 28, 28)
        elif image_array.shape[0] == 28 * 28:
            image_array = image_array.T.reshape(-1, 28, 28)
        else:
            raise ValueError(f'Unsupported 2D image shape: {image_array.shape}')
    elif image_array.ndim == 3:
        if image_array.shape[0] == 28 and image_array.shape[1] == 28:
            image_array = np.transpose(image_array, (2, 0, 1))
        elif image_array.shape[1] == 28 and image_array.shape[2] == 28:
            pass
        elif image_array.shape[0] == 28 and image_array.shape[2] == 28:
            image_array = np.transpose(image_array, (1, 0, 2))
        else:
            raise ValueError(f'Unsupported 3D image shape: {image_array.shape}')
    else:
        raise ValueError(f'Unsupported image rank: {image_array.ndim}')

    image_array = image_array.astype(np.float32)
    if image_array.max() <= 1.0:
        image_array = image_array * 255.0
    image_array = np.clip(image_array, 0, 255).astype(np.uint8)
    return image_array[:, None, :, :]


def standardize_labels(labels: Any, expected_num_classes: int) -> np.ndarray:
    # Convert labels into a 1D zero-based integer vector.
    label_array = np.asarray(labels).reshape(-1).astype(np.int64)

    min_label = int(label_array.min())
    max_label = int(label_array.max())
    if min_label == 1 and max_label == expected_num_classes:
        label_array = label_array - 1
    elif min_label != 0:
        label_array = label_array - min_label

    return label_array


def find_image_label_pairs(node: Any, path: str = 'root') -> List[Dict[str, Any]]:
    # Traverse the full object tree and extract valid (images, labels) pairs.
    pairs: List[Dict[str, Any]] = []

    if isinstance(node, dict):
        lowered = {str(key).lower(): value for key, value in node.items()}
        image_key = next((key for key in lowered if key in IMAGE_KEYS), None)
        label_key = next((key for key in lowered if key in LABEL_KEYS), None)

        if image_key is not None and label_key is not None:
            pairs.append(
                {
                    'path': path,
                    'images': lowered[image_key],
                    'labels': lowered[label_key],
                }
            )

        for key, value in node.items():
            pairs.extend(find_image_label_pairs(value, f'{path}.{key}'))
    elif isinstance(node, (list, tuple)):
        for index, value in enumerate(node):
            pairs.extend(find_image_label_pairs(value, f'{path}[{index}]'))

    return pairs


def load_emnist_subset(
    mat_path: Path,
    expected_num_classes: int,
    label_offset: int = 0,
) -> Tuple[np.ndarray, np.ndarray]:
    raw_data = load_mat_file(mat_path)
    candidate_pairs = find_image_label_pairs(raw_data)

    valid_pairs = []
    seen_signatures = set()
    for pair in candidate_pairs:
        try:
            images = standardize_images(pair['images'])
            labels = standardize_labels(pair['labels'], expected_num_classes)
        except Exception:
            continue

        if len(images) != len(labels):
            continue

        label_signature = int(labels[: min(256, len(labels))].sum())
        image_signature = int(images[:1].sum())
        signature = (
            images.shape,
            image_signature,
            len(labels),
            int(labels.min()),
            int(labels.max()),
            label_signature,
        )
        if signature in seen_signatures:
            continue

        seen_signatures.add(signature)
        valid_pairs.append({'path': pair['path'], 'images': images, 'labels': labels})

    if not valid_pairs:
        raise ValueError(f'Could not extract a valid dataset from {mat_path.name}')

    print(f'[{mat_path.name}] Detected {len(valid_pairs)} valid splits:')
    for pair in valid_pairs:
        print(f"  - {pair['path']}: {len(pair['labels'])} samples")

    all_images = np.concatenate([pair['images'] for pair in valid_pairs], axis=0)
    all_labels = np.concatenate([pair['labels'] for pair in valid_pairs], axis=0) + label_offset
    return all_images, all_labels


digits_images, digits_labels = load_emnist_subset(
    digits_path,
    expected_num_classes=cfg.NUM_CLASSES_DIGITS,
    label_offset=0,
)
letters_images, letters_labels = load_emnist_subset(
    letters_path,
    expected_num_classes=cfg.NUM_CLASSES_LETTERS,
    label_offset=cfg.NUM_CLASSES_DIGITS,
)

all_images = np.concatenate([digits_images, letters_images], axis=0)
all_labels = np.concatenate([digits_labels, letters_labels], axis=0)

print(f'Total samples after merging: {len(all_labels):,}')
print(f'Image tensor shape: {all_images.shape}')
print(f'Label range: [{all_labels.min()} - {all_labels.max()}]')

[emnist-digits.mat] Detected 2 valid splits:
  - root.dataset.train: 240000 samples
  - root.dataset.test: 40000 samples
[emnist-letters.mat] Detected 2 valid splits:
  - root.dataset.train: 124800 samples
  - root.dataset.test: 20800 samples
Total samples after merging: 425,600
Image tensor shape: (425600, 1, 28, 28)
Label range: [0 - 35]


In [ ]:
class EMNISTDataset(Dataset):
    # Lightweight dataset wrapper that converts images to float only inside __getitem__.
    def __init__(self, images: np.ndarray, labels: np.ndarray) -> None:
        self.images = torch.from_numpy(images)
        self.labels = torch.from_numpy(labels.astype(np.int64))

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        image = self.images[index].float() / 255.0
        label = self.labels[index].long()
        return image, label


def maybe_limit_indices(indices: np.ndarray, max_samples: Optional[int], seed: int) -> np.ndarray:
    if max_samples is None or len(indices) <= max_samples:
        return indices
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(indices, size=max_samples, replace=False))


def train_val_split_indices(total_size: int, val_ratio: float, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    indices = np.arange(total_size)
    rng.shuffle(indices)
    val_size = int(total_size * val_ratio)
    val_indices = indices[:val_size]
    train_indices = indices[val_size:]
    return train_indices, val_indices


def dirichlet_split_noniid(
    labels: np.ndarray,
    num_clients: int,
    alpha: float,
    min_size: int,
    seed: int,
) -> List[np.ndarray]:
    # Split data using a Dirichlet distribution to create realistic non-IID client partitions.
    unique_classes = np.unique(labels)
    rng = np.random.default_rng(seed)
    min_partition_size = 0

    while min_partition_size < min_size:
        client_relative_indices = [[] for _ in range(num_clients)]

        for class_id in unique_classes:
            class_indices = np.where(labels == class_id)[0]
            rng.shuffle(class_indices)

            proportions = rng.dirichlet(np.repeat(alpha, num_clients))
            split_points = (np.cumsum(proportions)[:-1] * len(class_indices)).astype(int)
            class_chunks = np.split(class_indices, split_points)

            for client_id, chunk in enumerate(class_chunks):
                client_relative_indices[client_id].extend(chunk.tolist())

        min_partition_size = min(len(indices) for indices in client_relative_indices)

    return [np.array(indices, dtype=np.int64) for indices in client_relative_indices]


def summarize_client_distribution(labels: np.ndarray, relative_indices: np.ndarray, top_k: int = 3) -> str:
    unique_labels, counts = np.unique(labels[relative_indices], return_counts=True)
    top_pairs = sorted(
        zip(unique_labels.tolist(), counts.tolist()),
        key=lambda item: item[1],
        reverse=True,
    )[:top_k]
    return ', '.join(f'{CLASS_NAMES[class_id]}: {count}' for class_id, count in top_pairs)


def build_client_partitions(
    labels: np.ndarray,
    train_indices: np.ndarray,
    config: Config,
) -> Tuple[Dict[int, np.ndarray], pd.DataFrame, np.ndarray]:
    train_labels = labels[train_indices]
    client_relative_splits = dirichlet_split_noniid(
        labels=train_labels,
        num_clients=config.NUM_CLIENTS,
        alpha=config.DIRICHLET_ALPHA,
        min_size=config.BATCH_SIZE,
        seed=config.RANDOM_SEED,
    )

    client_index_map: Dict[int, np.ndarray] = {}
    summary_rows = []
    client_class_matrix = np.zeros((config.NUM_CLIENTS, config.NUM_CLASSES), dtype=np.int64)

    for client_id, relative_indices in enumerate(client_relative_splits):
        absolute_indices = train_indices[relative_indices]
        client_index_map[client_id] = absolute_indices

        local_labels = labels[absolute_indices]
        client_class_matrix[client_id] = np.bincount(local_labels, minlength=config.NUM_CLASSES)
        summary_rows.append(
            {
                'Client ID': client_id,
                'Num Samples': len(absolute_indices),
                'Top Classes': summarize_client_distribution(train_labels, relative_indices),
            }
        )

    return client_index_map, pd.DataFrame(summary_rows), client_class_matrix


def build_client_loaders(
    dataset: Dataset,
    client_index_map: Dict[int, np.ndarray],
    config: Config,
) -> Dict[int, DataLoader]:
    client_loaders: Dict[int, DataLoader] = {}
    for client_id, absolute_indices in client_index_map.items():
        subset = Subset(dataset, absolute_indices.tolist())
        client_loaders[client_id] = DataLoader(
            subset,
            batch_size=config.BATCH_SIZE,
            shuffle=True,
            num_workers=config.NUM_WORKERS,
            pin_memory=device.type == 'cuda',
        )
    return client_loaders


full_dataset = EMNISTDataset(all_images, all_labels)
train_indices, val_indices = train_val_split_indices(len(full_dataset), cfg.VAL_RATIO, cfg.RANDOM_SEED)
train_indices = maybe_limit_indices(train_indices, cfg.MAX_TRAIN_SAMPLES, cfg.RANDOM_SEED)
val_cap = cfg.MC_VAL_SUBSET_SIZE if cfg.MAX_VAL_SAMPLES is None else min(cfg.MAX_VAL_SAMPLES, cfg.MC_VAL_SUBSET_SIZE or cfg.MAX_VAL_SAMPLES)
val_indices = maybe_limit_indices(val_indices, val_cap, cfg.RANDOM_SEED + 1)

client_index_map, client_distribution_df, client_class_matrix = build_client_partitions(
    labels=all_labels,
    train_indices=train_indices,
    config=cfg,
)
client_loaders = build_client_loaders(full_dataset, client_index_map, cfg)
val_loader = DataLoader(
    Subset(full_dataset, val_indices.tolist()),
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=device.type == 'cuda',
)

print(f'Federated training samples: {len(train_indices):,}')
print(f'Central validation samples: {len(val_indices):,}')
display(client_distribution_df)

Federated training samples: 383,040
Central validation samples: 4,000


,Client ID,Num Samples,Top Classes
0,0,18060,"3: 3391, 9: 2498, 0: 1355"
1,1,18394,"0: 5694, 3: 2965, 1: 1713"
2,2,15017,"9: 4539, 5: 3593, Y: 1320"
3,3,18394,"9: 5921, 6: 1627, 1: 1195"
4,4,24264,"3: 4352, 5: 3087, 0: 2472"
5,5,22136,"2: 5685, 9: 4499, 7: 2390"
6,6,13839,"2: 2267, 3: 1932, 6: 1858"
7,7,18801,"1: 7078, 7: 4068, 5: 961"
8,8,24118,"6: 4647, 7: 4128, 5: 3546"
9,9,26275,"4: 5425, 0: 4606, 2: 2401"


In [ ]:
class SimpleCNN(nn.Module):
    # A compact CNN with 2 Conv2d blocks, MaxPool layers, and 2 Linear layers.
    def __init__(self, num_classes: int) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        features = self.features(inputs)
        return self.classifier(features)


def clone_state_dict(state_dict: Dict[str, torch.Tensor]) -> OrderedDict:
    return OrderedDict(
        (key, value.detach().cpu().clone())
        for key, value in state_dict.items()
    )


def fedavg_state_dicts(weighted_states: List[Tuple[Dict[str, torch.Tensor], int]]) -> OrderedDict:
    # Weighted FedAvg aggregation by local sample count.
    if not weighted_states:
        raise ValueError('weighted_states must not be empty.')

    total_samples = sum(num_samples for _, num_samples in weighted_states)
    averaged_state = OrderedDict()
    for key in weighted_states[0][0].keys():
        weighted_sum = sum(
            state_dict[key].float() * (num_samples / total_samples)
            for state_dict, num_samples in weighted_states
        )
        averaged_state[key] = weighted_sum.clone()
    return averaged_state


def compute_model_size_mb(model: nn.Module) -> float:
    total_bytes = sum(tensor.numel() * tensor.element_size() for tensor in model.state_dict().values())
    return total_bytes / (1024 ** 2)


@torch.inference_mode()
def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    num_classes: int,
    return_confusion: bool = False,
):
    model.eval()
    correct_predictions = 0
    total_samples = 0
    confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64) if return_confusion else None

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        predictions = logits.argmax(dim=1)

        correct_predictions += (predictions == labels).sum().item()
        total_samples += labels.size(0)

        if confusion_matrix is not None:
            for true_label, pred_label in zip(labels.cpu().numpy(), predictions.cpu().numpy()):
                confusion_matrix[int(true_label), int(pred_label)] += 1

    accuracy = correct_predictions / max(total_samples, 1)
    if return_confusion:
        return accuracy, confusion_matrix
    return accuracy


def estimate_convergence_round(history: List[float], ratio: float) -> int:
    if not history:
        return 0
    target_accuracy = max(history) * ratio
    for round_index, accuracy in enumerate(history, start=1):
        if accuracy >= target_accuracy:
            return round_index
    return len(history)


def scenario_label(scenario) -> str:
    return f"{scenario.scenario_id} | {scenario.title}"

In [ ]:
@dataclass(frozen=True)
class ScenarioSpec:
    scenario_id: str
    title: str
    case_name: str
    mechanism: str
    attack_mode: str
    description: str


@dataclass(frozen=True)
class ClientBlueprint:
    client_id: int
    true_cost: float
    truthful_bid: float
    manipulated_bid: float
    is_bid_attacker: bool
    is_hacker: bool


class Client:
    def __init__(
        self,
        client_id: int,
        true_cost: float,
        truthful_bid: float,
        bid_price: float,
        config: Config,
        device: torch.device,
        is_hacker: bool = False,
        is_bid_attacker: bool = False,
    ) -> None:
        self.id = client_id
        self.true_cost = true_cost
        self.truthful_bid = truthful_bid
        self.bid_price = bid_price
        self.reputation = config.INITIAL_REPUTATION
        self.is_hacker = is_hacker
        self.is_bid_attacker = is_bid_attacker
        self.config = config
        self.device = device
        self.total_payment = 0.0
        self.last_payment = 0.0
        self.selected_rounds = 0
        self.accepted_rounds = 0

    def local_train(self, global_model: nn.Module, dataloader: DataLoader) -> Dict[str, Any]:
        # Each client receives the global model, trains locally, and returns a new state_dict.
        local_model = deepcopy(global_model).to(self.device)
        local_model.train()

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(local_model.parameters(), lr=self.config.LR, momentum=0.9)

        for _ in range(self.config.LOCAL_EPOCHS):
            for images, labels in dataloader:
                images = images.to(self.device)
                labels = labels.to(self.device)

                # Malicious clients use label flipping to corrupt their local update.
                if self.is_hacker:
                    labels = (labels + 1) % self.config.NUM_CLASSES

                optimizer.zero_grad()
                logits = local_model(images)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()

        return {
            'weights': clone_state_dict(local_model.state_dict()),
            'num_samples': len(dataloader.dataset),
        }


class VanillaSelectionCore:
    def __init__(self, budget: float, seed: int) -> None:
        self.budget = budget
        self.seed = seed

    def select_clients(self, clients: List[Client], round_idx: int) -> Tuple[List[Client], float]:
        # Baseline selection: random ready-order under a hard budget constraint.
        round_rng = random.Random(self.seed + round_idx)
        candidate_clients = clients.copy()
        round_rng.shuffle(candidate_clients)

        selected_clients: List[Client] = []
        spent_budget = 0.0
        for client in candidate_clients:
            if spent_budget + client.bid_price <= self.budget:
                selected_clients.append(client)
                spent_budget += client.bid_price
        return selected_clients, spent_budget


class EconomicCore:
    def __init__(self, budget: float) -> None:
        self.budget = budget

    def select_clients(
        self,
        clients: List[Client],
    ) -> Tuple[List[Client], Dict[int, float], Optional[Client], float]:
        # Myerson-style economic score: s_i = reputation / bid_price.
        ranked_clients = sorted(
            clients,
            key=lambda client: client.reputation / max(client.bid_price, 1e-8),
            reverse=True,
        )

        selected_clients: List[Client] = []
        first_loser: Optional[Client] = None
        spent_budget = 0.0

        for client in ranked_clients:
            if spent_budget + client.bid_price <= self.budget:
                selected_clients.append(client)
                spent_budget += client.bid_price
            elif first_loser is None:
                first_loser = client

        if not selected_clients:
            return [], {}, first_loser, spent_budget

        if first_loser is None:
            first_loser = selected_clients[-1]

        critical_ratio = first_loser.bid_price / max(first_loser.reputation, 1e-8)
        payments = {
            client.id: client.reputation * critical_ratio
            for client in selected_clients
        }
        return selected_clients, payments, first_loser, spent_budget


class SecurityCore:
    def __init__(
        self,
        model_factory,
        val_loader: DataLoader,
        device: torch.device,
        mc_iterations: int,
        num_classes: int,
    ) -> None:
        self.model_factory = model_factory
        self.val_loader = val_loader
        self.device = device
        self.mc_iterations = mc_iterations
        self.num_classes = num_classes
        self.accuracy_cache: Dict[Tuple[int, ...], float] = {}

    def _build_model(self) -> nn.Module:
        return self.model_factory().to(self.device)

    def evaluate(self, model: nn.Module) -> float:
        return evaluate_model(model, self.val_loader, self.device, self.num_classes)

    def _evaluate_state_dict(self, state_dict: Dict[str, torch.Tensor]) -> float:
        model = self._build_model()
        model.load_state_dict(state_dict)
        return self.evaluate(model)

    def _subset_accuracy(
        self,
        subset_ids: Tuple[int, ...],
        base_model_state: Dict[str, torch.Tensor],
        local_updates: Dict[int, Dict[str, Any]],
    ) -> float:
        # Cache subset evaluations to avoid recomputing the same merged model.
        sorted_subset = tuple(sorted(subset_ids))
        if sorted_subset in self.accuracy_cache:
            return self.accuracy_cache[sorted_subset]

        if not sorted_subset:
            accuracy = self._evaluate_state_dict(base_model_state)
        else:
            aggregated_state = fedavg_state_dicts(
                [
                    (local_updates[client_id]['weights'], local_updates[client_id]['num_samples'])
                    for client_id in sorted_subset
                ]
            )
            accuracy = self._evaluate_state_dict(aggregated_state)

        self.accuracy_cache[sorted_subset] = accuracy
        return accuracy

    def mc_sva(
        self,
        base_model_state: Dict[str, torch.Tensor],
        local_updates: Dict[int, Dict[str, Any]],
    ) -> Dict[int, float]:
        client_ids = list(local_updates.keys())
        shapley_scores = {client_id: 0.0 for client_id in client_ids}

        if not client_ids:
            return shapley_scores

        self.accuracy_cache = {}
        self.accuracy_cache[tuple()] = self._evaluate_state_dict(base_model_state)

        for _ in range(self.mc_iterations):
            permutation = client_ids.copy()
            random.shuffle(permutation)

            current_subset: List[int] = []
            previous_accuracy = self.accuracy_cache[tuple()]
            for client_id in permutation:
                candidate_subset = tuple(current_subset + [client_id])
                current_accuracy = self._subset_accuracy(candidate_subset, base_model_state, local_updates)
                shapley_scores[client_id] += current_accuracy - previous_accuracy
                current_subset.append(client_id)
                previous_accuracy = current_accuracy

        for client_id in shapley_scores:
            shapley_scores[client_id] /= self.mc_iterations

        return shapley_scores


class BaseFederatedServer:
    def __init__(
        self,
        config: Config,
        scenario: ScenarioSpec,
        clients: List[Client],
        client_loaders: Dict[int, DataLoader],
        val_loader: DataLoader,
        device: torch.device,
    ) -> None:
        self.config = config
        self.scenario = scenario
        self.clients = clients
        self.client_loaders = client_loaders
        self.val_loader = val_loader
        self.device = device
        self.global_model = SimpleCNN(config.NUM_CLASSES).to(device)
        self.model_size_mb = compute_model_size_mb(self.global_model)

        self.accuracy_history: List[float] = []
        self.communication_history_mb: List[float] = []
        self.cumulative_communication_history_mb: List[float] = []
        self.round_spending: List[float] = []
        self.selected_counts: List[int] = []
        self.accepted_counts: List[int] = []
        self.selection_history: List[List[int]] = []
        self.accepted_history: List[List[int]] = []
        self.round_extra_stats: List[Dict[str, Any]] = []
        self.shapley_history: List[Dict[int, float]] = []
        self.reputation_history: Dict[int, List[float]] = {
            client.id: [client.reputation]
            for client in clients
        }

        self.tracked_clean_id = next(
            (client.id for client in clients if not self._is_attacking_client(client)),
            None,
        )
        self.tracked_attack_id = next(
            (client.id for client in clients if self._is_attacking_client(client)),
            None,
        )

    def _is_attacking_client(self, client: Client) -> bool:
        if self.scenario.attack_mode == 'hacker':
            return client.is_hacker
        if self.scenario.attack_mode == 'bid_attack':
            return client.is_bid_attacker
        return False

    def _append_reputation_snapshot(self) -> None:
        for client in self.clients:
            self.reputation_history[client.id].append(client.reputation)

    def _should_log(self, round_idx: int) -> bool:
        return (
            round_idx == 1
            or round_idx % self.config.LOG_EVERY == 0
            or round_idx == self.config.GLOBAL_ROUNDS
        )

    def _apply_global_update(self, accepted_ids: List[int], local_updates: Dict[int, Dict[str, Any]]) -> None:
        if not accepted_ids:
            return
        aggregated_state = fedavg_state_dicts(
            [
                (local_updates[client_id]['weights'], local_updates[client_id]['num_samples'])
                for client_id in accepted_ids
            ]
        )
        self.global_model.load_state_dict(aggregated_state)

    def _compute_round_comm_mb(self, selected_count: int) -> float:
        # Communication = model downlink + local model uplink.
        return self.model_size_mb * selected_count * 2.0

    def _aggregate_numeric_extras(self) -> Dict[str, float]:
        totals: Dict[str, float] = {}
        for stats in self.round_extra_stats:
            for key, value in stats.items():
                if isinstance(value, (int, float, np.integer, np.floating)):
                    totals[key] = totals.get(key, 0.0) + float(value)
        return totals

    def _client_role(self, client: Client) -> str:
        if client.is_hacker:
            return 'Hacker'
        if client.is_bid_attacker:
            return 'Bid Attacker'
        return 'Clean'

    def build_client_summary_df(self) -> pd.DataFrame:
        rows = []
        for client in sorted(self.clients, key=lambda item: item.id):
            rows.append(
                {
                    'Client ID': client.id,
                    'Type': self._client_role(client),
                    'True Cost': round(client.true_cost, 4),
                    'Truthful Bid': round(client.truthful_bid, 4),
                    'Reported Bid': round(client.bid_price, 4),
                    'Total Payment': round(client.total_payment, 4),
                    'Final Reputation': round(client.reputation, 4),
                    'Selected Rounds': client.selected_rounds,
                    'Accepted Rounds': client.accepted_rounds,
                }
            )
        return pd.DataFrame(rows)

    def select_clients(self, round_idx: int):
        raise NotImplementedError

    def handle_round(self, round_idx: int, selected_clients: List[Client], local_updates: Dict[int, Dict[str, Any]], selection_info: Dict[str, Any]):
        raise NotImplementedError

    def run(self) -> Dict[str, Any]:
        start_time = time.time()
        progress_bar = tqdm(
            range(1, self.config.GLOBAL_ROUNDS + 1),
            desc=scenario_label(self.scenario),
            leave=True,
            dynamic_ncols=True,
        )

        for round_idx in progress_bar:
            selected_clients, selection_info = self.select_clients(round_idx)
            selected_ids = [client.id for client in selected_clients]

            local_updates: Dict[int, Dict[str, Any]] = {}
            for client in selected_clients:
                client.selected_rounds += 1
                local_updates[client.id] = client.local_train(self.global_model, self.client_loaders[client.id])

            accepted_ids, round_spending, extra_stats = self.handle_round(
                round_idx,
                selected_clients,
                local_updates,
                selection_info,
            )

            for client in selected_clients:
                if client.id in accepted_ids:
                    client.accepted_rounds += 1

            self._apply_global_update(accepted_ids, local_updates)

            round_comm_mb = self._compute_round_comm_mb(len(selected_clients))
            current_accuracy = evaluate_model(
                self.global_model,
                self.val_loader,
                self.device,
                self.config.NUM_CLASSES,
            )

            self.accuracy_history.append(current_accuracy)
            self.communication_history_mb.append(round_comm_mb)
            self.cumulative_communication_history_mb.append(float(np.sum(self.communication_history_mb)))
            self.round_spending.append(round_spending)
            self.selected_counts.append(len(selected_clients))
            self.accepted_counts.append(len(accepted_ids))
            self.selection_history.append(selected_ids)
            self.accepted_history.append(accepted_ids)
            self.round_extra_stats.append(extra_stats)
            self._append_reputation_snapshot()

            progress_bar.set_postfix(
                acc=f"{current_accuracy:.4f}",
                comm_mb=f"{self.cumulative_communication_history_mb[-1]:.2f}",
                selected=len(selected_ids),
                accepted=len(accepted_ids),
            )

            if self._should_log(round_idx):
                progress_bar.write('-' * 120)
                progress_bar.write(f"{scenario_label(self.scenario)} | Round {round_idx}/{self.config.GLOBAL_ROUNDS}")
                progress_bar.write(f"Selected clients: {selected_ids}")
                progress_bar.write(f"Accepted clients: {accepted_ids}")
                progress_bar.write(f"Validation accuracy: {current_accuracy:.4f}")
                progress_bar.write(f"Communication cost (round / cumulative MB): {round_comm_mb:.4f} / {self.cumulative_communication_history_mb[-1]:.4f}")
                progress_bar.write(f"Payment spent this round: {round_spending:.4f}")
                if 'hackers_caught' in extra_stats:
                    progress_bar.write(f"Hackers caught this round: {int(extra_stats['hackers_caught'])}")

        progress_bar.close()

        final_accuracy, final_confusion_matrix = evaluate_model(
            self.global_model,
            self.val_loader,
            self.device,
            self.config.NUM_CLASSES,
            return_confusion=True,
        )
        total_runtime_seconds = time.time() - start_time
        total_spending = float(np.sum(self.round_spending))
        total_comm_mb = float(np.sum(self.communication_history_mb))
        convergence_round = estimate_convergence_round(self.accuracy_history, self.config.CONVERGENCE_RATIO)

        return {
            'scenario': self.scenario,
            'scenario_label': scenario_label(self.scenario),
            'accuracy_history': self.accuracy_history,
            'communication_history_mb': self.communication_history_mb,
            'cumulative_communication_history_mb': self.cumulative_communication_history_mb,
            'round_spending': self.round_spending,
            'selected_counts': self.selected_counts,
            'accepted_counts': self.accepted_counts,
            'selection_history': self.selection_history,
            'accepted_history': self.accepted_history,
            'extra_stats_total': self._aggregate_numeric_extras(),
            'client_summary_df': self.build_client_summary_df(),
            'reputation_history': self.reputation_history,
            'tracked_clean_id': self.tracked_clean_id,
            'tracked_attack_id': self.tracked_attack_id,
            'final_accuracy': final_accuracy,
            'best_accuracy': max(self.accuracy_history) if self.accuracy_history else 0.0,
            'convergence_round': convergence_round,
            'total_communication_mb': total_comm_mb,
            'total_spending': total_spending,
            'model_size_mb': self.model_size_mb,
            'final_confusion_matrix': final_confusion_matrix,
            'shapley_history': self.shapley_history,
            'last_shapley_scores': self.shapley_history[-1] if self.shapley_history else {},
            'runtime_seconds': total_runtime_seconds,
        }


class VanillaFLServer(BaseFederatedServer):
    def __init__(
        self,
        config: Config,
        scenario: ScenarioSpec,
        clients: List[Client],
        client_loaders: Dict[int, DataLoader],
        val_loader: DataLoader,
        device: torch.device,
    ) -> None:
        super().__init__(config, scenario, clients, client_loaders, val_loader, device)
        self.selection_core = VanillaSelectionCore(config.BUDGET, config.RANDOM_SEED)

    def select_clients(self, round_idx: int):
        selected_clients, spent_budget = self.selection_core.select_clients(self.clients, round_idx)
        return selected_clients, {'spent_budget': spent_budget}

    def handle_round(self, round_idx: int, selected_clients: List[Client], local_updates: Dict[int, Dict[str, Any]], selection_info: Dict[str, Any]):
        accepted_ids = [client.id for client in selected_clients]
        round_spending = 0.0
        selected_attackers = 0
        selected_clean = 0

        for client in selected_clients:
            client.last_payment = client.bid_price
            client.total_payment += client.last_payment
            round_spending += client.last_payment
            if self._is_attacking_client(client):
                selected_attackers += 1
            else:
                selected_clean += 1

        extra_stats = {
            'selected_attackers': selected_attackers,
            'accepted_attackers': selected_attackers,
            'blocked_attackers': 0,
            'selected_clean': selected_clean,
            'accepted_clean': selected_clean,
            'blocked_clean': 0,
            'hackers_caught': 0,
        }
        return accepted_ids, round_spending, extra_stats


class TRASServer(BaseFederatedServer):
    def __init__(
        self,
        config: Config,
        scenario: ScenarioSpec,
        clients: List[Client],
        client_loaders: Dict[int, DataLoader],
        val_loader: DataLoader,
        device: torch.device,
    ) -> None:
        super().__init__(config, scenario, clients, client_loaders, val_loader, device)
        self.economic_core = EconomicCore(config.BUDGET)
        self.security_core = SecurityCore(
            model_factory=lambda: SimpleCNN(config.NUM_CLASSES),
            val_loader=val_loader,
            device=device,
            mc_iterations=config.MC_ITERATIONS,
            num_classes=config.NUM_CLASSES,
        )

    def select_clients(self, round_idx: int):
        selected_clients, payments, first_loser, spent_budget = self.economic_core.select_clients(self.clients)
        return selected_clients, {
            'payments': payments,
            'first_loser': first_loser.id if first_loser is not None else None,
            'spent_budget': spent_budget,
        }

    def handle_round(self, round_idx: int, selected_clients: List[Client], local_updates: Dict[int, Dict[str, Any]], selection_info: Dict[str, Any]):
        payments = selection_info.get('payments', {})
        base_model_state = clone_state_dict(self.global_model.state_dict())
        shapley_scores = self.security_core.mc_sva(base_model_state, local_updates)
        self.shapley_history.append(shapley_scores.copy())

        accepted_ids = [client_id for client_id, score in shapley_scores.items() if score > 0.0]
        selected_id_set = {client.id for client in selected_clients}
        round_spending = 0.0
        selected_attackers = 0
        selected_clean = 0
        accepted_attackers = 0
        accepted_clean = 0
        blocked_attackers = 0
        blocked_clean = 0

        for client in self.clients:
            if client.id not in selected_id_set:
                client.last_payment = 0.0
                continue

            if self._is_attacking_client(client):
                selected_attackers += 1
            else:
                selected_clean += 1

            shapley_value = shapley_scores.get(client.id, 0.0)
            if shapley_value > 0.0:
                client.last_payment = float(payments.get(client.id, 0.0))
                client.total_payment += client.last_payment
                client.reputation += self.config.REPUTATION_REWARD
                round_spending += client.last_payment
                if self._is_attacking_client(client):
                    accepted_attackers += 1
                else:
                    accepted_clean += 1
            else:
                client.last_payment = 0.0
                client.reputation = max(0.0, client.reputation - self.config.REPUTATION_PENALTY)
                if self._is_attacking_client(client):
                    blocked_attackers += 1
                else:
                    blocked_clean += 1

        extra_stats = {
            'selected_attackers': selected_attackers,
            'accepted_attackers': accepted_attackers,
            'blocked_attackers': blocked_attackers,
            'selected_clean': selected_clean,
            'accepted_clean': accepted_clean,
            'blocked_clean': blocked_clean,
            'hackers_caught': blocked_attackers if self.scenario.attack_mode == 'hacker' else 0,
        }
        return accepted_ids, round_spending, extra_stats


def build_client_blueprints(config: Config) -> List[ClientBlueprint]:
    rng = np.random.default_rng(config.RANDOM_SEED)
    client_ids = np.arange(config.NUM_CLIENTS)
    num_hackers = max(1, int(config.NUM_CLIENTS * config.HACKER_RATIO))
    num_bid_attackers = max(1, int(config.NUM_CLIENTS * config.BID_ATTACK_RATIO))

    hacker_ids = set(rng.choice(client_ids, size=num_hackers, replace=False).tolist())
    bid_attacker_ids = set(rng.choice(client_ids, size=num_bid_attackers, replace=False).tolist())

    blueprints: List[ClientBlueprint] = []
    for client_id in client_ids:
        true_cost = float(rng.uniform(4.0, 8.0))
        truthful_bid = true_cost
        manipulated_bid = float(true_cost * config.BID_ATTACK_MULTIPLIER)
        blueprints.append(
            ClientBlueprint(
                client_id=int(client_id),
                true_cost=true_cost,
                truthful_bid=truthful_bid,
                manipulated_bid=manipulated_bid,
                is_bid_attacker=int(client_id) in bid_attacker_ids,
                is_hacker=int(client_id) in hacker_ids,
            )
        )
    return blueprints


def instantiate_clients_for_scenario(
    blueprints: List[ClientBlueprint],
    scenario: ScenarioSpec,
    config: Config,
    device: torch.device,
) -> List[Client]:
    clients: List[Client] = []

    for blueprint in blueprints:
        is_hacker = scenario.attack_mode == 'hacker' and blueprint.is_hacker
        is_bid_attacker = scenario.attack_mode == 'bid_attack' and blueprint.is_bid_attacker

        # In the dishonest bidding case, the baseline uses manipulated bids while T-RAS uses truthful bids.
        if scenario.attack_mode == 'bid_attack' and scenario.mechanism == 'Vanilla FL':
            reported_bid = blueprint.manipulated_bid if blueprint.is_bid_attacker else blueprint.truthful_bid
        else:
            reported_bid = blueprint.truthful_bid

        clients.append(
            Client(
                client_id=blueprint.client_id,
                true_cost=blueprint.true_cost,
                truthful_bid=blueprint.truthful_bid,
                bid_price=reported_bid,
                config=config,
                device=device,
                is_hacker=is_hacker,
                is_bid_attacker=is_bid_attacker,
            )
        )

    return clients

In [ ]:
blueprints = build_client_blueprints(cfg)
blueprint_df = pd.DataFrame(
    [
        {
            'Client ID': blueprint.client_id,
            'True Cost': round(blueprint.true_cost, 4),
            'Truthful Bid': round(blueprint.truthful_bid, 4),
            'Manipulated Bid': round(blueprint.manipulated_bid, 4),
            'Bid Attacker': blueprint.is_bid_attacker,
            'Hacker': blueprint.is_hacker,
        }
        for blueprint in blueprints
    ]
)
display(blueprint_df)

scenario_specs = [
    ScenarioSpec(
        scenario_id='S1',
        title='Clean Data + Vanilla FL',
        case_name='Clean',
        mechanism='Vanilla FL',
        attack_mode='clean',
        description='Clean data, no attack, baseline budgeted FedAvg.',
    ),
    ScenarioSpec(
        scenario_id='S2',
        title='Clean Data + T-RAS',
        case_name='Clean',
        mechanism='T-RAS',
        attack_mode='clean',
        description='Clean data, no attack, using EconomicCore + SecurityCore.',
    ),
    ScenarioSpec(
        scenario_id='S3',
        title='Dishonest Bidding + Vanilla FL',
        case_name='Dishonest Bid Attack',
        mechanism='Vanilla FL',
        attack_mode='bid_attack',
        description='Malicious bidders over-report cost and reduce budget efficiency under the baseline.',
    ),
    ScenarioSpec(
        scenario_id='S4',
        title='Dishonest Bidding + T-RAS',
        case_name='Dishonest Bid Attack',
        mechanism='T-RAS',
        attack_mode='bid_attack',
        description='T-RAS simulates truthful bidding under the Myerson-style mechanism.',
    ),
    ScenarioSpec(
        scenario_id='S5',
        title='Hacker Poisoning + Vanilla FL',
        case_name='Hacker Poisoning',
        mechanism='Vanilla FL',
        attack_mode='hacker',
        description='Hackers perform label flipping while the baseline has no dedicated defense.',
    ),
    ScenarioSpec(
        scenario_id='S6',
        title='Hacker Poisoning + T-RAS',
        case_name='Hacker Poisoning',
        mechanism='T-RAS',
        attack_mode='hacker',
        description='T-RAS uses MC-SVA to filter harmful model updates.',
    ),
]

scenario_results: Dict[str, Dict[str, Any]] = OrderedDict()
benchmark_rows = []

for scenario in scenario_specs:
    print('\n' + '#' * 120)
    print(f"STARTING {scenario_label(scenario)}")
    print(scenario.description)
    print('#' * 120)

    clients = instantiate_clients_for_scenario(blueprints, scenario, cfg, device)
    if scenario.mechanism == 'Vanilla FL':
        server = VanillaFLServer(cfg, scenario, clients, client_loaders, val_loader, device)
    else:
        server = TRASServer(cfg, scenario, clients, client_loaders, val_loader, device)

    result = server.run()
    scenario_results[scenario.scenario_id] = result

    extra_stats_total = result['extra_stats_total']
    benchmark_rows.append(
        {
            'Scenario ID': scenario.scenario_id,
            'Scenario': scenario.title,
            'Case': scenario.case_name,
            'Mechanism': scenario.mechanism,
            'Final Accuracy': round(result['final_accuracy'], 6),
            'Best Accuracy': round(result['best_accuracy'], 6),
            'Convergence Round': int(result['convergence_round']),
            'Total Communication (MB)': round(result['total_communication_mb'], 4),
            'Total Spending': round(result['total_spending'], 4),
            'Avg Selected Clients / Round': round(float(np.mean(result['selected_counts'])), 4),
            'Avg Accepted Clients / Round': round(float(np.mean(result['accepted_counts'])), 4),
            'Blocked Attackers': int(extra_stats_total.get('blocked_attackers', 0)),
            'Accepted Attackers': int(extra_stats_total.get('accepted_attackers', 0)),
            'Runtime (s)': round(result['runtime_seconds'], 2),
        }
    )

benchmark_df = pd.DataFrame(benchmark_rows).sort_values('Scenario ID').reset_index(drop=True)
print('\nBENCHMARK SUMMARY TABLE')
display(benchmark_df)

,Client ID,True Cost,Truthful Bid,Manipulated Bid,Bid Attacker,Hacker
0,0,5.8015,5.8015,13.9237,False,False
1,1,5.4832,5.4832,13.1597,True,True
2,2,7.7071,7.7071,18.4969,False,False
3,3,6.5755,6.5755,15.7811,True,False
4,4,7.2910,7.2910,17.4985,False,False
5,5,5.7737,5.7737,13.8568,False,False
6,6,4.9090,4.9090,11.7815,False,False
7,7,6.2183,6.2183,14.9240,False,False
8,8,4.2553,4.2553,10.2126,False,True
9,9,7.3105,7.3105,17.5453,True,False



########################################################################################################################
STARTING S1 | Clean Data + Vanilla FL
Clean data, no attack, baseline budgeted FedAvg.
########################################################################################################################


S1 | Clean Data + Vanilla FL:   0%|          | 0/100 [00:00<?, ?it/s]

------------------------------------------------------------------------------------------------------------------------
S1 | Clean Data + Vanilla FL | Round 1/100
Selected clients: [8, 13, 5, 2, 6, 3, 17, 0, 16, 18, 7, 19, 12, 15, 10, 11]
Accepted clients: [8, 13, 5, 2, 6, 3, 17, 0, 16, 18, 7, 19, 12, 15, 10, 11]
Validation accuracy: 0.7268
Communication cost (round / cumulative MB): 51.8794 / 51.8794
Payment spent this round: 94.6515
Hackers caught this round: 0
------------------------------------------------------------------------------------------------------------------------
S1 | Clean Data + Vanilla FL | Round 2/100
Selected clients: [8, 14, 15, 10, 2, 12, 7, 9, 11, 1, 19, 0, 4, 18, 6, 5]
Accepted clients: [8, 14, 15, 10, 2, 12, 7, 9, 11, 1, 19, 0, 4, 18, 6, 5]
Validation accuracy: 0.8562
Communication cost (round / cumulative MB): 51.8794 / 103.7588
Payment spent this round: 97.2051
Hackers caught this round: 0
-----------------------------------------------------------------

S2 | Clean Data + T-RAS:   0%|          | 0/100 [00:00<?, ?it/s]

------------------------------------------------------------------------------------------------------------------------
S2 | Clean Data + T-RAS | Round 1/100
Selected clients: [18, 8, 19, 16, 6, 12, 1, 5, 0, 17, 7, 10, 3, 11, 15, 4, 9]
Accepted clients: [18, 19, 16, 6, 12, 5, 0, 17, 7, 10, 3, 11, 15, 4, 9]
Validation accuracy: 0.7095
Communication cost (round / cumulative MB): 55.1219 / 55.1219
Payment spent this round: 113.5873
Hackers caught this round: 0
------------------------------------------------------------------------------------------------------------------------
S2 | Clean Data + T-RAS | Round 2/100
Selected clients: [18, 19, 16, 6, 12, 5, 0, 17, 7, 10, 3, 11, 15, 4, 9, 8, 1]
Accepted clients: [19, 6, 12, 5, 0, 17, 7, 10, 3, 15, 9, 8, 1]
Validation accuracy: 0.8702
Communication cost (round / cumulative MB): 55.1219 / 110.2437
Payment spent this round: 100.7140
Hackers caught this round: 0
----------------------------------------------------------------------------------

S3 | Dishonest Bidding + Vanilla FL:   0%|          | 0/100 [00:00<?, ?it/s]

------------------------------------------------------------------------------------------------------------------------
S3 | Dishonest Bidding + Vanilla FL | Round 1/100
Selected clients: [8, 13, 5, 2, 6, 3, 17, 0, 16, 18, 7, 19, 12, 15]
Accepted clients: [8, 13, 5, 2, 6, 3, 17, 0, 16, 18, 7, 19, 12, 15]
Validation accuracy: 0.7542
Communication cost (round / cumulative MB): 45.3945 / 45.3945
Payment spent this round: 96.1435
Hackers caught this round: 0
------------------------------------------------------------------------------------------------------------------------
S3 | Dishonest Bidding + Vanilla FL | Round 2/100
Selected clients: [8, 14, 15, 10, 2, 12, 7, 9, 11, 19, 0]
Accepted clients: [8, 14, 15, 10, 2, 12, 7, 9, 11, 19, 0]
Validation accuracy: 0.8715
Communication cost (round / cumulative MB): 35.6671 / 81.0616
Payment spent this round: 99.5466
Hackers caught this round: 0
----------------------------------------------------------------------------------------------------

In [ ]:
print('Detailed client table for scenario S4 - Dishonest Bidding + T-RAS')
display(scenario_results['S4']['client_summary_df'])

print('Detailed client table for scenario S6 - Hacker Poisoning + T-RAS')
display(scenario_results['S6']['client_summary_df'])

In [ ]:
palette = {
    'S1': '#1f77b4',
    'S2': '#ff7f0e',
    'S3': '#2ca02c',
    'S4': '#d62728',
    'S5': '#9467bd',
    'S6': '#8c564b',
}


def build_metrics_dataframe(results: Dict[str, Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for scenario_id, result in results.items():
        scenario = result['scenario']
        rows.append(
            {
                'Scenario ID': scenario_id,
                'Scenario Label': result['scenario_label'],
                'Case': scenario.case_name,
                'Mechanism': scenario.mechanism,
                'Final Accuracy': result['final_accuracy'],
                'Best Accuracy': result['best_accuracy'],
                'Convergence Round': result['convergence_round'],
                'Total Communication (MB)': result['total_communication_mb'],
                'Total Spending': result['total_spending'],
                'Avg Selected Clients': float(np.mean(result['selected_counts'])),
                'Avg Accepted Clients': float(np.mean(result['accepted_counts'])),
                'Blocked Attackers': result['extra_stats_total'].get('blocked_attackers', 0.0),
            }
        )
    return pd.DataFrame(rows).sort_values('Scenario ID').reset_index(drop=True)


metrics_df = build_metrics_dataframe(scenario_results)
display(metrics_df)

# ======================================================
# Figure 1: Accuracy curve and cumulative communication
# ======================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
rounds = np.arange(1, cfg.GLOBAL_ROUNDS + 1)
for scenario_id, result in scenario_results.items():
    axes[0].plot(rounds, result['accuracy_history'], label=scenario_id, color=palette[scenario_id], linewidth=2)
    axes[1].plot(rounds, result['cumulative_communication_history_mb'], label=scenario_id, color=palette[scenario_id], linewidth=2)

axes[0].set_title('Accuracy across 100 rounds for all six scenarios')
axes[0].set_xlabel('Round')
axes[0].set_ylabel('Validation Accuracy')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].set_title('Cumulative communication cost across 100 rounds')
axes[1].set_xlabel('Round')
axes[1].set_ylabel('Communication (MB)')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# ======================================================
# Figure 2: Bar charts for key metrics
# ======================================================
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
sns.barplot(data=metrics_df, x='Scenario ID', y='Final Accuracy', palette=palette, ax=axes[0])
sns.barplot(data=metrics_df, x='Scenario ID', y='Convergence Round', palette=palette, ax=axes[1])
sns.barplot(data=metrics_df, x='Scenario ID', y='Total Communication (MB)', palette=palette, ax=axes[2])
axes[0].set_title('Final Accuracy')
axes[1].set_title('Convergence Round')
axes[2].set_title('Total Communication Cost (MB)')
for axis in axes:
    axis.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# ======================================================
# Figure 3: Heatmap of normalized metrics
# ======================================================
heatmap_df = metrics_df.set_index('Scenario ID')[
    ['Final Accuracy', 'Best Accuracy', 'Convergence Round', 'Total Communication (MB)', 'Total Spending']
].copy()
normalized_df = heatmap_df.copy()
for column in normalized_df.columns:
    column_values = normalized_df[column].astype(float)
    if column in {'Convergence Round', 'Total Communication (MB)', 'Total Spending'}:
        normalized_df[column] = 1.0 - (column_values - column_values.min()) / max(column_values.max() - column_values.min(), 1e-8)
    else:
        normalized_df[column] = (column_values - column_values.min()) / max(column_values.max() - column_values.min(), 1e-8)

plt.figure(figsize=(10, 6))
sns.heatmap(normalized_df, annot=heatmap_df.round(4), fmt='', cmap='YlGnBu', cbar_kws={'label': 'Normalized Score'})
plt.title('Metric matrix: raw values and normalized scores')
plt.ylabel('Scenario')
plt.xlabel('Metric')
plt.tight_layout()
plt.show()

# ======================================================
# Figure 4: Scatter plot of final accuracy vs communication
# ======================================================
plt.figure(figsize=(9, 6))
for _, row in metrics_df.iterrows():
    marker = 'o' if row['Mechanism'] == 'Vanilla FL' else 's'
    plt.scatter(
        row['Total Communication (MB)'],
        row['Final Accuracy'],
        s=220,
        marker=marker,
        color=palette[row['Scenario ID']],
        alpha=0.85,
    )
    plt.text(
        row['Total Communication (MB)'] + 0.01,
        row['Final Accuracy'] + 0.001,
        row['Scenario ID'],
        fontsize=10,
    )
plt.title('Scatter plot: Final Accuracy vs Communication Cost')
plt.xlabel('Total Communication (MB)')
plt.ylabel('Final Accuracy')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# ======================================================
# Figure 5: Bid histogram and Shapley histogram
# ======================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].hist(
    scenario_results['S3']['client_summary_df']['Reported Bid'],
    bins=10,
    alpha=0.65,
    label='S3 - Vanilla FL',
    color=palette['S3'],
)
axes[0].hist(
    scenario_results['S4']['client_summary_df']['Reported Bid'],
    bins=10,
    alpha=0.65,
    label='S4 - T-RAS',
    color=palette['S4'],
)
axes[0].set_title('Histogram of reported bids in the dishonest bidding case')
axes[0].set_xlabel('Reported Bid')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

for scenario_id in ['S2', 'S4', 'S6']:
    shapley_values = list(scenario_results[scenario_id]['last_shapley_scores'].values())
    if shapley_values:
        axes[1].hist(shapley_values, bins=10, alpha=0.55, label=scenario_id, color=palette[scenario_id])
axes[1].set_title('Histogram of final-round Shapley values in T-RAS scenarios')
axes[1].set_xlabel('Shapley Value')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# ======================================================
# Figure 6: S6 pie chart and client-class heatmap
# ======================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
s6_breakdown = scenario_results['S6']['extra_stats_total']
pie_labels = [
    'Clean accepted',
    'Hackers blocked',
    'Clean blocked',
    'Hackers accepted',
]
pie_values = [
    s6_breakdown.get('accepted_clean', 0.0),
    s6_breakdown.get('blocked_attackers', 0.0),
    s6_breakdown.get('blocked_clean', 0.0),
    s6_breakdown.get('accepted_attackers', 0.0),
]
non_zero_pairs = [(label, value) for label, value in zip(pie_labels, pie_values) if value > 0]
axes[0].pie(
    [pair[1] for pair in non_zero_pairs],
    labels=[pair[0] for pair in non_zero_pairs],
    autopct='%1.1f%%',
    startangle=90,
)
axes[0].set_title('Scenario S6: accepted vs blocked updates')

sns.heatmap(client_class_matrix, cmap='mako', ax=axes[1])
axes[1].set_title('Non-IID client-class distribution heatmap')
axes[1].set_xlabel('Class ID')
axes[1].set_ylabel('Client ID')
plt.tight_layout()
plt.show()

# ======================================================
# Figure 7: Confusion matrix for the best scenario
# ======================================================
best_scenario_id = metrics_df.sort_values('Best Accuracy', ascending=False).iloc[0]['Scenario ID']
best_result = scenario_results[best_scenario_id]
plt.figure(figsize=(12, 10))
sns.heatmap(best_result['final_confusion_matrix'], cmap='crest', square=True)
plt.title(f"Confusion matrix for the best scenario: {best_scenario_id}")
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

# ======================================================
# Figure 8: Reputation trace for S4 and S6
# ======================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for axis, scenario_id in zip(axes, ['S4', 'S6']):
    result = scenario_results[scenario_id]
    tracked_clean_id = result['tracked_clean_id']
    tracked_attack_id = result['tracked_attack_id']
    reputation_rounds = np.arange(0, len(result['reputation_history'][tracked_clean_id]))

    axis.plot(
        reputation_rounds,
        result['reputation_history'][tracked_clean_id],
        label=f'Clean client #{tracked_clean_id}',
        linewidth=2,
        marker='o',
    )
    if tracked_attack_id is not None:
        axis.plot(
            reputation_rounds,
            result['reputation_history'][tracked_attack_id],
            label=f'Attack client #{tracked_attack_id}',
            linewidth=2,
            marker='o',
        )
    axis.set_title(f'Reputation trace - {scenario_id}')
    axis.set_xlabel('Round (0 = before training)')
    axis.set_ylabel('Reputation')
    axis.legend()
    axis.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# ======================================================
# Figure 9: Client selection frequency in S6
# ======================================================
s6_client_df = scenario_results['S6']['client_summary_df'].copy()
plt.figure(figsize=(12, 6))
sns.barplot(data=s6_client_df, x='Client ID', y='Selected Rounds', hue='Type')
plt.title('Client selection frequency in scenario S6')
plt.xlabel('Client ID')
plt.ylabel('Selected Rounds')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()